# Module 6: All Beauty data cleaning
**Student:** Wayne J. Chen
**Branch:** cleaning-waynechen

Read `docs/cleaning_rules.md` before running. This notebook runs the complete category, not the preliminary 1 MiB sample. Raw files are immutable, and 50 genuine records are shared as an audit sample. Every output comes from the Python scripts; restart the kernel and run all cells to regenerate. The team vision is prioritized over the older Appliances proposal.

In [1]:
from pathlib import Path
import sys, subprocess, json
import pandas as pd
ROOT = Path.cwd()
assert (ROOT / 'source_manifest.json').exists(), 'Open the notebook from cleaning_waynechen/'
print('Python:', sys.version)
print('Rules:', (ROOT / 'docs/cleaning_rules.md').read_text()[:1500])

def run_script(name):
    result = subprocess.run([sys.executable, 'scripts/' + name], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode:
        raise RuntimeError(result.stderr)

Python: 3.12.14 (main, Aug 25 2026, 14:00:49) [Clang 22.1.3 ]
Rules: # Cleaning rules declared before running the pipeline

Scope: full All_Beauty reviews and full All_Beauty product metadata, following the newer project vision PDF. The older branch's Appliances proposal is retained in docs/project-start.md for provenance only.

Business question: Which All Beauty parent listings should a category manager review first for possible missing or unclear information raised repeatedly in customer reviews? Main future comparison: number of distinct retained review records mentioning a theme per parent listing, with at least two reviews per theme and available metadata. This assignment prepares data; it does not assert that keyword mentions prove information gaps.

| Rule | Reason | Check |
|---|---|---|
| Keep raw downloads unchanged; pin the HF revision and verify SHA-256 and file sizes before parsing. Fail on invalid JSON. | Preserve provenance and catch interrupted downloads. | Input hashe

In [2]:
run_script('download_data.py')

Verified existing /workspace/scratch/ef2f90b6a9d6/deliverables/cleaning_waynechen/data/raw/All_Beauty.jsonl
Verified existing /workspace/scratch/ef2f90b6a9d6/deliverables/cleaning_waynechen/data/raw/meta_All_Beauty.jsonl



In [3]:
run_script('clean_data.py')
run_script('verify_outputs.py')

Processing reviews
Processing metadata
{
  "reviews": 694253,
  "metadata": 112590,
  "merge": {
    "joined_rows": 694253,
    "matched_rows": 694253,
    "unmatched_review_rows": 0,
    "unmatched_parent_keys": 0,
    "matched_parent_keys": 112565,
    "missing_parent_key_rows": 0,
    "eligible_metadata_rows": 112590,
    "metadata_without_reviews": 25
  },
  "metrics": {
    "review_count": 694253,
    "rating_n": 694253,
    "rating_sum": 2749862,
    "average_rating": 3.960893219042626,
    "negative_review_n": 143489,
    "negative_review_rate": 0.20668113785608416,
    "purchase_n": 694253,
    "verified_purchase_n": 628456,
    "verified_purchase_rate": 0.9052261927568191,
    "helpful_n": 694253,
    "helpful_vote_sum": 641610,
    "review_body_n": 693548,
    "in_scope_date_n": 694253,
    "parent_product_n": 112565,
    "reviewer_n": 631986,
    "date_min": "2000-11-01T04:24:18.000+00:00",
    "date_max": "2023-09-09T00:39:36.666+00:00"
  }
}

{
  "passed": true,
  "indepen

In [4]:
audit = json.loads((ROOT / 'outputs/audit.json').read_text())
print(json.dumps({k: {x: v[x] for x in ['raw_rows', 'exact_duplicate_rows_removed', 'retained_rows']} for k, v in audit['tables'].items()}, indent=2))
print(json.dumps(audit['merge'], indent=2))
print(json.dumps(audit['metrics'], indent=2))

{
  "metadata": {
    "raw_rows": 112590,
    "exact_duplicate_rows_removed": 0,
    "retained_rows": 112590
  },
  "reviews": {
    "raw_rows": 701528,
    "exact_duplicate_rows_removed": 7275,
    "retained_rows": 694253
  }
}
{
  "eligible_metadata_rows": 112590,
  "joined_rows": 694253,
  "matched_parent_keys": 112565,
  "matched_rows": 694253,
  "metadata_without_reviews": 25,
  "missing_parent_key_rows": 0,
  "unmatched_parent_keys": 0,
  "unmatched_review_rows": 0
}
{
  "average_rating": 3.960893219042626,
  "date_max": "2023-09-09T00:39:36.666+00:00",
  "date_min": "2000-11-01T04:24:18.000+00:00",
  "helpful_n": 694253,
  "helpful_vote_sum": 641610,
  "in_scope_date_n": 694253,
  "negative_review_n": 143489,
  "negative_review_rate": 0.20668113785608416,
  "parent_product_n": 112565,
  "purchase_n": 694253,
  "rating_n": 694253,
  "rating_sum": 2749862,
  "review_body_n": 693548,
  "review_count": 694253,
  "reviewer_n": 631986,
  "verified_purchase_n": 628456,
  "verified_purc

In [5]:
column_audit = pd.read_csv(ROOT / 'outputs/column_audit.csv')
display(column_audit)

,table,column,raw_missing,retained_preclean_missing,cleaned_missing,raw_types,cleaned_types,changed_retained_values,failed_conversion,invalid_domain
0,reviews,asin,0,0,0,"{""str"":701528}","{""str"":694253}",0,0,0
1,reviews,helpful_vote,0,0,0,"{""int"":701528}","{""int"":694253}",0,0,0
2,reviews,images,641844,635269,635269,"{""list"":701528}","{""list"":694253}",0,0,0
3,reviews,parent_asin,0,0,0,"{""str"":701528}","{""str"":694253}",0,0,0
4,reviews,rating,0,0,0,"{""float"":701528}","{""int"":694253}",694253,0,0
5,reviews,text,107,106,705,"{""str"":701528}","{""NoneType"":705,""str"":693548}",53270,0,0
6,reviews,timestamp,0,0,0,"{""int"":701528}","{""int"":694253}",0,0,0
7,reviews,title,0,0,0,"{""str"":701528}","{""str"":694253}",154,0,0
8,reviews,user_id,0,0,0,"{""str"":701528}","{""str"":694253}",0,0,0
9,reviews,verified_purchase,0,0,0,"{""bool"":701528}","{""bool"":694253}",0,0,0


In [6]:
checks = json.loads((ROOT / 'outputs/record_checks.json').read_text())
assert all(c['matches'] for c in checks)
display(pd.DataFrame([{k: c[k] for k in ['case', 'source_line', 'original', 'expected', 'actual', 'matches']} for c in checks]))

,case,source_line,original,expected,actual,matches
0,HTML line breaks preserve words and typo,37,{'text': 'Rrain hats are really handy when<br ...,{'text': 'Rrain hats are really handy when one...,{'text': 'Rrain hats are really handy when one...,True
1,HTML ampersand preserves doubled spaces and wo...,19599,{'text': 'Great stuff and I'm a vey satisfied ...,{'text': 'Great stuff and I'm a vey satisfied ...,{'text': 'Great stuff and I'm a vey satisfied ...,True
2,Missing body retains valid rating,15160,"{'text': '', 'rating': 5.0}","{'text': None, 'rating': 5}","{'text': None, 'rating': 5}",True
3,Leading-zero ASIN remains string,9585,"{'asin': '0816091846', 'parent_asin': '0816091...","{'asin': '0816091846', 'parent_asin': '0816091...","{'asin': '0816091846', 'parent_asin': '0816091...",True
4,Missing listing price is not imputed as zero,1,"{'parent_asin': 'B00YQ6X8EO', 'rating': 5.0, '...","{'parent_asin': 'B00YQ6X8EO', 'rating': 5, 'me...","{'parent_asin': 'B00YQ6X8EO', 'rating': 5, 'me...",True


In [7]:
sample = pd.read_csv(ROOT / 'outputs/samples/cleaned_sample.csv', dtype={'asin':'string','parent_asin':'string','user_id':'string'})
assert len(sample) == 50
display(sample[['record_id','parent_asin','rating','metadata_matched','sample_reason']].head(12))
print('Five real records and all 50 sample rows verified. IDs remain strings.')

Five real records and all 50 sample rows verified. IDs remain strings.


,record_id,parent_asin,rating,metadata_matched,sample_reason
0,All_Beauty:0000037,B00023J4AW,5,True,Five-record check: HTML line breaks preserve w...
1,All_Beauty:0019599,B00KCTER3U,5,True,Five-record check: HTML ampersand preserves do...
2,All_Beauty:0015160,B075RB3XJV,5,True,Five-record check: Missing body retains valid ...
3,All_Beauty:0009585,0816091846,5,True,Five-record check: Leading-zero ASIN remains s...
4,All_Beauty:0000001,B00YQ6X8EO,5,True,Five-record check: Missing listing price is no...
5,All_Beauty:0013846,B09NS1VG4L,5,True,repeated candidate review key
6,All_Beauty:0000004,B09JS339BZ,1,True,low rating
7,All_Beauty:0000025,B07ZJKVVLW,5,True,high helpful votes
8,All_Beauty:0622810,B08CM5931J,3,True,Deterministic SHA256 rank of record_id
9,All_Beauty:0618107,B01N7XE0A4,5,True,Deterministic SHA256 rank of record_id


## Interpretation and limitations
Missing listing content remains missing: a missing price is not zero, and an empty review body does not invalidate a known rating. Exact duplicates are removed; distinct repeated candidate keys remain flagged. All Beauty contains taxonomy anomalies such as numeric-looking book/movie products. The historical data cannot establish current listing gaps, causal business impact, or sales. Review proposed changes and five record checks yourself before submitting. Run `python scripts/reproduce.py` for the independent process/hash comparison; see the README for the final commit-ID step.